# Retrieving data from Github

In [5]:
#Using the requests library to download the data from the evidently github repo
import requests

repo_owner = 'evidentlyai'
repo_name = 'docs'
branch_name = 'main'

zip_url = f'https://github.com/{repo_owner}/{repo_name}/archive/refs/heads/{branch_name}.zip'
zip_response = requests.get(zip_url)

In [6]:
#The response content in bytes
len(zip_response.content)

17545754

In [7]:
#Opening the file as a ZIP without saving to disk
import io
import zipfile

zip_archive = zipfile.ZipFile(io.BytesIO(zip_response.content))

In [8]:
filenames = zip_archive.namelist()
filenames[20:30]

['docs-main/docs/library/prompt_optimization.mdx',
 'docs-main/docs/library/report.mdx',
 'docs-main/docs/library/synthetic_data_api.mdx',
 'docs-main/docs/library/tags_metadata.mdx',
 'docs-main/docs/library/tests.mdx',
 'docs-main/docs/platform/',
 'docs-main/docs/platform/alerts.mdx',
 'docs-main/docs/platform/dashboard_add_panels.mdx',
 'docs-main/docs/platform/dashboard_add_panels_ui.mdx',
 'docs-main/docs/platform/dashboard_overview.mdx']

In [9]:
#Reading one of the files
filename = 'docs-main/docs/platform/alerts.mdx'
mdx_file = zip_archive.open(filename)
mdx_content = mdx_file.read().decode('utf-8')
print(mdx_content[:150])

---
title: 'Alerts'
description: 'How to set up alerts.'
---

<Check>
  Built-in alerting is a Pro feature available in the **Evidently Cloud** and **


In [10]:
#Parsing metadata
import frontmatter

post = frontmatter.loads(mdx_content)
print(post.content[:100])

<Check>
  Built-in alerting is a Pro feature available in the **Evidently Cloud** and **Evidently En


In [11]:
#Removing file prefix
filename_corrected = filename.split('/', 1)[-1]
print(filename_corrected)

docs/platform/alerts.mdx


In [12]:
doc = {
    'content': post.content,
    'title': post.metadata.get('title'),
    'description': post.metadata.get('description'),
    'filename': filename_corrected
}

In [13]:
#A function to ingest/feed documentation into the system before indexing or querying it
def read_github_repository(repo_owner, repo_name, branch="main"):
    url = f"https://github.com/{repo_owner}/{repo_name}/archive/refs/heads/{branch}.zip"
    response = requests.get(url)
    response.raise_for_status()

    documents = []
    with zipfile.ZipFile(io.BytesIO(response.content)) as zip_ref:
        for file_path in zip_ref.namelist():
            if not file_path.endswith(('.md', '.mdx')):
                continue
            with zip_ref.open(file_path) as file:
                content = file.read().decode('utf-8')
                post = frontmatter.loads(content)
                doc = {
                    'content': post.content,
                    'title': post.metadata.get('title'),
                    'description': post.metadata.get('description'),
                    'filename': file_path.split('/', 1)[-1]
                }
                documents.append(doc)

    return documents

In [14]:
repo_owner = 'evidentlyai'
repo_name = 'docs'

documents = read_github_repository(repo_owner, repo_name)

print(f"Downloaded {len(documents)} documents")

Downloaded 95 documents


In [15]:
!uv add gitsource

Resolved 119 packages in 28ms
Checked 116 packages in 39ms


In [16]:
#Downloading the same documentation with gitsource
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)

files = reader.read()

print(f"Loaded {len(files)} documents")

Loaded 95 documents


In [17]:
document = files[10]

print(document.filename)
print(document.content[:160])

docs/library/output_formats.mdx
---
title: 'Output formats'
description: 'How to export the evaluation results.'
---

You can view or export Reports in multiple formats.

**Pre-requisites**:




In [18]:
document.parse()

{'title': 'Output formats',
 'description': 'How to export the evaluation results.',
 'content': 'You can view or export Reports in multiple formats.\n\n**Pre-requisites**:\n\n* You know how to [generate Reports](/docs/library/report).\n\n## Log to Workspace\n\nYou can save the computed Report in Evidently Cloud or your local workspace.\n\n```python\nws.add_run(project.id, my_eval, include_data=False)\n```\n\n<Info>\n  **Uploading evals**. Check Quickstart examples [for ML](/quickstart_ml) or [for LLM](/quickstart_llm) for a full workflow.\n</Info>\n\n## View in Jupyter notebook\n\nYou can directly render the visual summary of evaluation results in interactive Python environments like Jupyter notebook or Colab.\n\nAfter running the Report, simply call the resulting Python object:\n\n```python\nmy_report\n```\n\nThis will render the HTML object directly in the notebook cell.\n\n## HTML\n\nYou can also save this interactive visual Report as an HTML file to open in a browser:\n\n```python

In [19]:
#Preparing the documents for indexing
documents = [f.parse() for f in files]

In [20]:
len(documents)

95

# SEARCH

In [21]:
!uv add minsearch

Resolved 119 packages in 2ms
Checked 116 packages in 3ms


In [22]:
from minsearch import Index

In [23]:
#text_fields - fields to search with full-text search
#keyword_fields - fields for exact matching only
index = Index(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(documents)

In [24]:
#Testing if its searching within the defined fields
query = 'LLM as a Judge'
results = index.search(query, num_results=5)

In [25]:
len(results)

5

# CHUNKING

In [26]:
doc_sizes = [(doc.filename, len(doc.content)) for doc in files]
doc_sizes.sort(key=lambda x: x[1], reverse=True)

for filename, size in doc_sizes[:5]:
    print(f"{filename}: {size} characters")

metrics/all_metrics.mdx: 55085 characters
metrics/all_descriptors.mdx: 31976 characters
docs/platform/dashboard_panel_types.mdx: 31647 characters
docs/library/leftover_content.mdx: 28742 characters
metrics/customize_llm_judge.mdx: 26847 characters


In [27]:
#Applying sliding window to our data
def sliding_window(text, size=1000, step=500):
    chunks = []
    start = 0
    text_length = len(text)

    while start < text_length:
        end = start + size
        chunk = text[start:end]
        chunks.append({'start': start, 'content': chunk})

        start = end - step

        if end >= text_length:
            break

    return chunks

In [28]:
len(sliding_window(results[0]['content'], size=3000, step=2500))

39

In [29]:
#Adding indexes to the chunks
document_chunks = []

for doc in documents:
    if not doc.get('content'):
        continue
    copy = doc.copy()
    content = copy.pop('content')

    chunks = sliding_window(content, size=3000, step=1500)

    for i, chunk in enumerate(chunks):
        chunk.update(copy)
        chunk['chunk_id'] = i
        document_chunks.append(chunk)

In [30]:
document_chunks[10]

{'start': 9000,
 'content': 'cation=[BinaryClassification(\n        target="target",\n        prediction_labels="prediction")],\n    categorical_columns=["target", "prediction"])\n```\n\nAvailable options and defaults:\n\n```python\n    target: str = "target"\n    prediction_labels: Optional[str] = None\n    prediction_probas: Optional[str] = "prediction" #if probabilistic classification\n    pos_label: Label = 1 #name of the positive label\n    labels: Optional[Dict[Label, str]] = None\n```\n\n### Ranking\n\n#### RecSys\n\nTo evaluate recommender systems performance, you must map the columns with:\n\n- Prediction: this could be predicted score or rank.\n- Target: relevance labels (e.g., this could be an interaction result like user click or upvote, or a true relevance label)\n\nThe **target** column can contain either:\n\n- a binary label (where `1` is a positive outcome)\n- any scores (positive values, where a higher value corresponds to a better match or a more valuable user action)

In [31]:
document_chunks[2]

{'start': 1500,
 'content': 'ently v0.7.8">\n  ## **Evidently 0.7.8**\n\n  Full release notes on [Github](https://github.com/evidentlyai/evidently/releases/tag/v0.7.8).\n</Update>\n\n<Update label="2025-06-04" description="Evidently v0.7.7">\n  ## **Evidently 0.7.7**\n\n  Full release notes on [Github](https://github.com/evidentlyai/evidently/releases/tag/v0.7.7).\n</Update>\n\n<Update label="2025-05-25" description="Evidently v0.7.6">\n  ## **Evidently 0.7.6**\n\n  Full release notes on [Github](https://github.com/evidentlyai/evidently/releases/tag/v0.7.6).\n</Update>\n\n<Update label="2025-05-09" description="Evidently v0.7.5">\n  ## **Evidently 0.7.5**\n\n  Full release notes on [Github](https://github.com/evidentlyai/evidently/releases/tag/v0.7.5).\n</Update>\n\n<Update label="2025-05-05" description="Evidently v0.7.4">\n  ## **Evidently 0.7.4**\n\n  Full release notes on [Github](https://github.com/evidentlyai/evidently/releases/tag/v0.7.4).\n</Update>\n\n<Update label="2025-04-25

In [32]:
#Creating an index for the chunks
chunk_index = Index(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
chunk_index.fit(document_chunks)

In [33]:
results = chunk_index.search(query)

In [34]:
len(results)

10

In [35]:
from gitsource import chunk_documents

document_chunks = chunk_documents(documents, size=3000, step=1500)

In [36]:
document_chunks[2]

{'start': 1500,
 'content': 'ently v0.7.8">\n  ## **Evidently 0.7.8**\n\n  Full release notes on [Github](https://github.com/evidentlyai/evidently/releases/tag/v0.7.8).\n</Update>\n\n<Update label="2025-06-04" description="Evidently v0.7.7">\n  ## **Evidently 0.7.7**\n\n  Full release notes on [Github](https://github.com/evidentlyai/evidently/releases/tag/v0.7.7).\n</Update>\n\n<Update label="2025-05-25" description="Evidently v0.7.6">\n  ## **Evidently 0.7.6**\n\n  Full release notes on [Github](https://github.com/evidentlyai/evidently/releases/tag/v0.7.6).\n</Update>\n\n<Update label="2025-05-09" description="Evidently v0.7.5">\n  ## **Evidently 0.7.5**\n\n  Full release notes on [Github](https://github.com/evidentlyai/evidently/releases/tag/v0.7.5).\n</Update>\n\n<Update label="2025-05-05" description="Evidently v0.7.4">\n  ## **Evidently 0.7.4**\n\n  Full release notes on [Github](https://github.com/evidentlyai/evidently/releases/tag/v0.7.4).\n</Update>\n\n<Update label="2025-04-25

# RAG

In [37]:
from anthropic import Anthropic
anthropic_client = Anthropic()

In [38]:
search_result = chunk_index.search(query, num_results=5)

In [39]:
len(search_result)

5

In [40]:
query = "How to implement LLM as a judge"

In [41]:
import json

search_result_json = json.dumps(search_result, indent=2)

In [42]:
instructions = """
You're a course assistant, your task is to answer the QUESTION from the
course students using the provided CONTEXT
"""

user_prompt = f"""
<QUESTION>
{query}
</QUESTION>

<CONTEXT>
{search_result_json}
</CONTEXT>
""".strip()

In [43]:
def llm(user_prompt, instructions=None, model='claude-haiku-4-5'):

    messages = []

    # if instructions is not None:
    #     messages.append({
    #         "role": "assistant",
    #         "content": instructions
    #     })

    messages.append({
        "role": "user",
        "content": user_prompt
    })

    response = anthropic_client.messages.create(
        model=model,
        max_tokens=1000,
        system=instructions,
        messages=messages
    )

    return response.content

In [44]:
answer = llm(user_prompt, instructions)

In [45]:
print(answer)

[TextBlock(citations=None, text='# How to Implement LLM as a Judge\n\nBased on the course materials, here\'s a comprehensive guide to implementing LLM as a judge:\n\n## Overview\n\nThere are **two main approaches**:\n\n1. **Reference-based**: Compare new responses against a reference/ground truth\n2. **Open-ended**: Evaluate responses based on custom criteria without a reference\n\n## Step-by-Step Implementation\n\n### 1. **Installation and Setup**\n\n```python\npip install evidently\n```\n\nImport required modules:\n```python\nimport pandas as pd\nfrom evidently import Dataset, Report\nfrom evidently.descriptors import *\nfrom evidently.llm.templates import BinaryClassificationPromptTemplate\n\n# Set your OpenAI API key\nimport os\nos.environ["OPENAI_API_KEY"] = "YOUR_KEY"\n```\n\n### 2. **Create Your Evaluation Dataset**\n\nBuild a dataset with:\n- Questions/inputs\n- Target/reference responses\n- New responses to evaluate\n- Manual labels (for later validation)\n\n### 3. **Define th

In [46]:
def search(query):
    return chunk_index.search(query, num_results=5)

In [47]:
instructions = """
You're a course assistant, your task is to answer the QUESTION from the
course students using the provided CONTEXT
"""

def build_prompt(query, search_results):
    search_result_json = json.dumps(search_results, indent=2)

    user_prompt = f"""
    <QUESTION>
    {query}
    </QUESTION>

    <CONTEXT>
    {search_result_json}
    </CONTEXT>
    """.strip()

    return user_prompt

In [48]:
def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt, instructions)
    return answer

In [49]:
rag("how do i setup alerts")

[TextBlock(citations=None, text='# Setting Up Alerts\n\nTo set up alerts in Evidently, follow these steps:\n\n## 1. Navigate to Alerts\nOpen your Project and go to the **"Alerts"** option in the left menu.\n\n## 2. Configure a Notification Channel\nYou must choose at least one notification channel:\n\n- **Email** - Add email addresses to receive alerts\n- **Slack** - Add a Slack webhook\n- **Discord** - Add a Discord webhook\n\n## 3. Set Alert Conditions\n\nYou can alert on two types of conditions:\n\n### Option A: Failed Tests\nIf you\'re using Tests (conditional checks) in your Project, you can enable alerts for failed Tests in a Test Suite. When any Test fails, Evidently will send an alert to your notification channel.\n\n**Tip to avoid alert fatigue:** Use the `is_critical` parameter set to `False` to mark non-critical Tests as Warnings. These won\'t trigger alerts even if they fail.\n\n### Option B: Custom Conditions on Metrics\nYou can set alerts on individual Metric values. For 

In [50]:
rag('how do i customize my LLM judge')

[TextBlock(citations=None, text='# Customizing Your LLM Judge\n\nBased on the tutorial, here are the key ways to customize your LLM judge:\n\n## 1. **Create a Custom Prompt Template**\n\nDefine a custom evaluation template using `BinaryClassificationPromptTemplate`:\n\n```python\nfrom evidently.llm.templates import BinaryClassificationPromptTemplate\n\nverbosity = BinaryClassificationPromptTemplate(\n    name="verbosity",\n    description="Check if the response is concise",\n    criteria="""Determine whether the response is concise or verbose. \n    A concise response conveys the necessary information without unnecessary elaboration. \n    A verbose response adds extra information that is not directly relevant or makes \n    the message unnecessarily long without improving how the message effectively.""",\n    target_category="concise",\n    non_target_category="verbose",\n    uncertainty="unknown",\n    include_reasoning=True,\n    pre_messages=[(("system", "You are an expert text eva

## Structured output
### The most reliable way to force the LLM to output what you want is through the structured output schema, not just the system prompt.

In [53]:
from pydantic import BaseModel

class CalendarEvent(BaseModel):
    name: str
    date: str
    participants: list[str]

In [134]:
def llm_structured(user_prompt, output_type, instructions=None, model='claude-haiku-4-5'):

    messages = []

    # if instructions is not None:
    #     messages.append({
    #         "role": "assistant",
    #         "content": instructions
    #     })

    messages.append({
        "role": "user",
        "content": user_prompt
    })

    response = anthropic_client.messages.parse(
        model=model,
        max_tokens=1000,
        messages=messages,
        system=instructions if instructions else "",
        output_format=output_type
    )

    return response.parsed_output

In [135]:
response = llm_structured(
    instructions="Extract the event information.",
    user_prompt="Alice and Bob are going to a science fair on Friday.",
    output_type=CalendarEvent
)

In [136]:
print(response)

name='Science Fair' date='Friday' participants=['Alice', 'Bob']


In [137]:
from typing import Optional
from pydantic import Field
from typing import Literal

class RAGResponse(BaseModel):
    """
    The response from the documentation RAG system
    """
    answer: Optional[str] = Field(None, description="Answer the question based on the CONTEXT from our documentation. If you don't find the answer, set `answer` to None")
    found_answer: bool = Field(description="True if relevant information was found in the documentation")
    confidence: float = Field(description="Confidence score from 0.0 to 1.0 indicating how certain the answer is")
    confidence_explanation: str = Field(description="Explanation about the confidence level")
    answer_type: Literal["how-to", "explanation", "troubleshooting", "comparison", "reference"] = Field(description="The category of the answer")
    followup_questions: list[str] = Field(description="Suggested follow-up questions the user might want to ask")

In [138]:
RAGResponse.model_json_schema()

{'description': 'The response from the documentation RAG system',
 'properties': {'answer': {'anyOf': [{'type': 'string'}, {'type': 'null'}],
   'default': None,
   'description': "Answer the question based on the CONTEXT from our documentation. If you don't find the answer, set `answer` to None",
   'title': 'Answer'},
  'found_answer': {'description': 'True if relevant information was found in the documentation',
   'title': 'Found Answer',
   'type': 'boolean'},
  'confidence': {'description': 'Confidence score from 0.0 to 1.0 indicating how certain the answer is',
   'title': 'Confidence',
   'type': 'number'},
  'confidence_explanation': {'description': 'Explanation about the confidence level',
   'title': 'Confidence Explanation',
   'type': 'string'},
  'answer_type': {'description': 'The category of the answer',
   'enum': ['how-to',
    'explanation',
    'troubleshooting',
    'comparison',
    'reference'],
   'title': 'Answer Type',
   'type': 'string'},
  'followup_questio

In [139]:
def rag_structured(query, output_type=RAGResponse):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    return llm_structured(
        instructions=instructions,
        user_prompt=prompt,
        output_type=output_type,
    )

In [126]:
answer = rag_structured('how do I do llm evals?', RAGResponse)

print(answer.answer)
print(answer.confidence)
print(answer.found_answer)

To do LLM evals, you can use Evidently's evaluation framework. Here's a quick guide:

1. **Install and Import**: Install Evidently with `pip install evidently[llm]` and import the necessary modules (Dataset, DataDefinition, descriptors, etc.)

2. **Prepare Your Data**: Create a pandas DataFrame with your evaluation data (questions, responses, contexts, etc.)

3. **Choose Evaluation Type**:
   - **Retrieval Quality**: Use `ContextQualityLLMEval()` to assess if retrieved context can answer the question
   - **Context Relevance**: Use `ContextRelevance()` to check if chunks contain relevant information
   - **Generation with Ground Truth**: Use `CorrectnessLLMEval()` to compare responses against target answers, or `SemanticSimilarity()` and `BERTScore()` for non-LLM methods
   - **Generation without Ground Truth**: Use `FaithfulnessLLMEval()` to detect if responses contradict the context

4. **Create Dataset with Descriptors**: Create an Evidently Dataset object and specify your descripto

In [140]:
answer = rag_structured('how do I install kafka on windows?', RAGResponse)

print(answer.answer)
print(answer.found_answer)

None
False


In [148]:
from pydantic import model_validator

class AnswerNotFound(BaseModel):
    explanation: str

class AnswerResponse(BaseModel):
    """
     Mutually exclusive fields — set EXACTLY ONE, never both, never neither:
    - Set 'answer' if a valid answer was found in the context.
    - Set 'answer_not_found' if no answer could be found. Leave 'answer' as null.
    """
    answer: Optional[RAGResponse] = None
    answer_not_found: Optional[AnswerNotFound] = None

    @model_validator(mode="after")
    def check_consistency(self):
        if self.answer is not None and self.answer_not_found is not None:
            raise ValueError("Provide either 'answer' or 'answer_not_found', not both.")
        if self.answer is None and self.answer_not_found is None:
            raise ValueError("Provide either 'answer' or 'answer_not_found'.")
        return self

    @property
    def found_answer(self) -> bool:
        return self.answer is not None

In [149]:
AnswerResponse.model_json_schema()

{'$defs': {'AnswerNotFound': {'properties': {'explanation': {'title': 'Explanation',
     'type': 'string'}},
   'required': ['explanation'],
   'title': 'AnswerNotFound',
   'type': 'object'},
  'RAGResponse': {'description': 'The response from the documentation RAG system',
   'properties': {'answer': {'anyOf': [{'type': 'string'}, {'type': 'null'}],
     'default': None,
     'description': "Answer the question based on the CONTEXT from our documentation. If you don't find the answer, set `answer` to None",
     'title': 'Answer'},
    'found_answer': {'description': 'True if relevant information was found in the documentation',
     'title': 'Found Answer',
     'type': 'boolean'},
    'confidence': {'description': 'Confidence score from 0.0 to 1.0 indicating how certain the answer is',
     'title': 'Confidence',
     'type': 'number'},
    'confidence_explanation': {'description': 'Explanation about the confidence level',
     'title': 'Confidence Explanation',
     'type': 'stri

In [150]:
answer = rag_structured('how do I install kafka on windows?', AnswerResponse)
answer

AnswerResponse(answer=None, answer_not_found=AnswerNotFound(explanation='The provided documentation context is about Evidently, a Python library for AI evaluations, and does not contain any information about installing Kafka on Windows. The documentation covers installation of Evidently, running evaluations, synthetic data generation, and exploring evaluation results, but does not include Kafka installation instructions.'))

In [151]:
answer = rag_structured('how do I run llm evals?', AnswerResponse)
answer

AnswerResponse(answer=RAGResponse(answer=None, found_answer=True, confidence=0.95, confidence_explanation='The documentation provides clear, step-by-step instructions for running LLM evals with code examples and workflow details.', answer_type='how-to', followup_questions=['How do I upload my evaluation results to the platform?', 'What are descriptors and how do I compute them?', 'How do I configure test conditions for my evals?', 'How do I explore and compare evaluation results?', 'What are the data requirements for running evals?', 'Can I customize the TextEvals report?', 'How do I set up monitoring and alerts for my evaluations?', "What's the difference between running evals locally vs uploading to the platform?"]), answer_not_found=None)

## Error handling

In [152]:
#Sometimes the LLM might generate both fields as None, we can catch and retry 

from pydantic import ValidationError

try:
    AnswerResponse()
except ValidationError as e:
    print("Validation error:")
    print(e)

Validation error:
1 validation error for AnswerResponse
  Value error, Provide either 'answer' or 'answer_not_found'. [type=value_error, input_value={}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error
